# JeevaSwara / KanthaRakshak: 04. Model Baseline & Patient-Level Cross-Validation

In this notebook, we benchmark standard scikit-learn classification algorithms:
1. Logistic Regression
2. Random Forest
3. Support Vector Machine (RBF kernel)
4. Gradient Boosting

### Zero-Leakage Validation Policy:
We strictly employ `StratifiedGroupKFold` grouped on `patient_code` so that all sessions belonging to any single patient appear strictly in train OR validation, never both.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))
from app.pipeline.model_trainer import benchmark_models_cv, NUMERICAL_FEATURE_COLUMNS

df = pd.read_csv("../data/research_cohort_sessions.csv")
print(f"Benchmarking models across {df['patient_code'].nunique()} patient groups...")

results = benchmark_models_cv(df, features=NUMERICAL_FEATURE_COLUMNS, n_splits=5)
metrics_table = []
for model_name, dat in results.items():
    agg = dat["aggregated"]
    metrics_table.append({
        "Model": model_name,
        "Sensitivity (Recall)": f"{agg['sensitivity_mean']:.3f} ± {agg['sensitivity_std']:.3f}",
        "Specificity": f"{agg['specificity_mean']:.3f} ± {agg['specificity_std']:.3f}",
        "Precision": f"{agg['precision_mean']:.3f} ± {agg['precision_std']:.3f}",
        "F1-Score": f"{agg['f1_mean']:.3f} ± {agg['f1_std']:.3f}",
        "ROC-AUC": f"{agg['roc_auc_mean']:.3f} ± {agg['roc_auc_std']:.3f}"
    })

pd.DataFrame(metrics_table)


### Model Comparison Visualizer
Comparing mean Sensitivity and Specificity across candidate algorithms.


In [ ]:
models = list(results.keys())
sens = [results[m]["aggregated"]["sensitivity_mean"] for m in models]
spec = [results[m]["aggregated"]["specificity_mean"] for m in models]
f1 = [results[m]["aggregated"]["f1_mean"] for m in models]

x = np.arange(len(models))
width = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x - width, sens, width, label='Sensitivity (Recall)', color='#0d9488')
plt.bar(x, spec, width, label='Specificity', color='#0284c7')
plt.bar(x + width, f1, width, label='F1 Score', color='#6366f1')

plt.xticks(x, models, rotation=15)
plt.ylabel('Score (0.0 to 1.0)')
plt.ylim(0.7, 1.05)
plt.title('Patient-Level Cross-Validation Performance (StratifiedGroupKFold)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


### Cross-Validation Summary
- **Random Forest** and **Logistic Regression** demonstrate robust generalization without patient data leakage.
- In swallow screening decision support, **minimizing false negatives (maximizing sensitivity)** is paramount to prevent missed aspiration risks.
